In [1]:
import sys
from pathlib import Path

from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
    
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form


/home/zagar/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose the prompting configuration

In [17]:
filename = "1989CanLII1415ONCA"
split = "test"
filepath = Path(DATA_DIR) / "original" / split / f"{filename}.html"
filepath = Path("output") / f"{filename}_processed2.html"
with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()

In [18]:
### Choose the right worflow
method = "DEC3" # "AIO" | "DEC0" | "DEC1" | "DEC2" | "DEC3"

if method == "AIO":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = []
    new_labels = ["decision", "legislation", "secondary sources", "title", "citation", "source", "authors", "fragment"]

    spans_in_context = True

    prompt_filename = "allInOne_long.txt"


if method == "DEC0":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = []
    new_labels = ["decision", "legislation", "secondary sources"]

    spans_in_context = True

    prompt_filename = "decomposed0_short.txt"

if method == "DEC1":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = ["decision", "legislation", "secondary sources"]
    new_labels = ["title", "fragment"]

    spans_in_context = False


    prompt_filename = "decomposed1-3.txt"

if method == "DEC2":
    parents = ["secondary sources"]
    already_labeled_labels = ["decision", "legislation", "secondary sources", "title", "fragment"]
    new_labels = ["source", "authors"]

    spans_in_context = False

    prompt_filename = "decomposed1-3.txt"

if method == "DEC3":
    parents = ["decision", "legislation"]
    already_labeled_labels = ["decision", "legislation", "secondary sources", "title", "fragment", "source", "authors"]
    new_labels = ["citation"]

    spans_in_context = False

    prompt_filename = "decomposed1-3.txt"

#### Commun Few SHot Selection

In [19]:
fewshot_method = "greedy"   # "greedy" | "random"

with open(FEWSHOT_CACHE_DIR / f"examples_{fewshot_method}.json", "r", encoding="utf-8") as f:
    fewshot_file_content = json.load(f)

fewshot_examples = [(example["example"]["input"], example["example"]["output"]) for example in fewshot_file_content["examples"]]



##### Few Shot processing step

In [20]:
nb_fewshot_examples = 6
allowed_labels = already_labeled_labels + new_labels

input_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=already_labeled_labels,
    keep_attributes=["labelname"]
)

output_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=already_labeled_labels + new_labels,
    keep_attributes=["labelname"]
)


# Transform the output in it simplified form
final_fewshot = []
total_output_text = ""
for example in fewshot_examples:
    input, output = example

    input_tokens = tokenize(input)
    transformed_input_tokens = prepare_label_tokens(input_tokens, input_label_config)

    output_tokens = tokenize(output)
    transformed_output_tokens = prepare_label_tokens(output_tokens, output_label_config)

    if spans_in_context:
        final_fewshot.append((decode(transformed_input_tokens), decode(transformed_output_tokens)))

    if not spans_in_context:

        total_output_text += "|||" + decode(transformed_output_tokens)

final_fewshot = final_fewshot[:nb_fewshot_examples]


if not spans_in_context:
    parents_dict = _parse_parent_annotations(total_output_text)
    for parent_name, annotations in parents_dict.items():
        if parent_name not in parents:
            continue
        for annotation in annotations:
            input = decode(prepare_label_tokens(simplified_to_normal_form(tokenize(annotation),label_type="manual_label"), input_label_config))

            if input != annotation:
                final_fewshot.append((input, annotation))

#### Commun Prompt loading

In [21]:
from src.prompts.prompt_utils import build_sublabel_definitions

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

from src.prompts.sublabel_definitions import SUBLABEL_DEFINITIONS_V2


if method in ["DEC1", "DEC2", "DEC3"]:
    sublabels_str = ", ".join(new_labels)
    sublabels_definition = build_sublabel_definitions(set(new_labels) - set(parents), sublabel_definitions=SUBLABEL_DEFINITIONS_V2)

    system_prompt = system_prompt.format(
            sublabels=sublabels_str,
            sublabels_definition=sublabels_definition,
        )

system_prompt used :  decomposed1-3.txt


#### Assistant loading

In [7]:
from src.models import AssistantFactory

#gpt5_2_config= {
#        "type": "openai",
#        "model_name": "gpt-5.2",
#        "temperature": 1,
#    }

#assistant = AssistantFactory.create_from_config(gpt5_2_config)



assistant = AssistantFactory.create("Qwen2.5-7B-Instruct")

Loading Qwen2.5-7B-Instruct from /home/zagar/scratch/Qwen2.5-7B-Instruct [fp16]


2026-06-05 14:06:21.589604: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-05 14:06:23.742812: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780693583.950878 1098063 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780693584.032565 1098063 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780693584.496116 1098063 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

#### Chunk output controle

In [ ]:
def process_output(generated, token_chunk, allowed_labels, assistant, with_fallback: bool = True):

    from src.output_control.processor import OutputProcessor
    from src.output_control.fallback import FallbackHandler 
    from src.output_control.verification import VerificationResult 

    controller = OutputProcessor()
    fallback_handler = FallbackHandler(processor=controller)
    
    corrected_generated_tokens, status = controller.process(
        raw_llm_output=generated,
        original_chunk=token_chunk,
        allowed_labels=allowed_labels
    )

    if status.passed:
        return token_chunk, status

    if not with_fallback:
        return token_chunk, status
    
    
    corrected_generated_tokens, status_dict = fallback_handler.handle_failure(
        assistant=assistant,
        corrected_output=corrected_generated_tokens,
        original_chunk=token_chunk,
        initial_status=status,
        allowed_labels=allowed_labels,
        fallback_prompt_filename="fallback.txt"
    )
    # Convert dict to VerificationResult
    status = VerificationResult(
        passed=status_dict.get('passed', False),
        error_type=status_dict.get('error_type'),
        details=status_dict.get('error_details'),
        tokens=corrected_generated_tokens
    )
    
    return corrected_generated_tokens, status

### For AIO or DEC0 ONLY

#### Chunking with the chunker

In [137]:
chunker = "sentence"  # "paragraph" | "sentence"

from src.chunkers.cache import cache_exists, load_cache
from src.chunkers import ChunkerFactory

if not cache_exists(chunker, split, filename):

    # Load spaCy only if needed
    nlp = None
    if chunker == "sentence":
        import spacy
        nlp = spacy.load("en_core_web_trf")
        print("✅ Model loaded.\n")


    token_chunks = ChunkerFactory.get_chunks(
        html_content, method=chunker, split=split, filename=filename, nlp=nlp
    )

else:
    token_chunks = load_cache(chunker, split, filename)




In [140]:
token_chunk = token_chunks[0]

In [141]:
user_input =  decode(token_chunk)

In [142]:
message = get_message(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=True)


In [143]:
generated = generate(assistant=assistant, messages=message)


In [144]:
print(generated)

Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the Excise Tax Act,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the Excise Tax Act. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the Excise Tax Act. <legislation>Schedule III of the Excise Tax Act</legislation>. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.
Member W.
Roy Hines W. Roy
Hines
Member
Robert J. Martin Robert J. Martin
Secretary 

In [145]:
corrected_generated_tokens, status = process_output(generated, token=token_chunk, allowed_labels=allowed_labels, assistant=assistant)


In [146]:
print(decode(corrected_generated_tokens))

 Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the Excise Tax Act,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the Excise Tax Act. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
<auto_label labelname="legislation">Schedule III of the Excise Tax Act</auto_label>. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.
Member W.
Roy Hines W. Roy
Hines
Member
Robert J. Martin Robert J. Martin
Secretary UNOFFICIAL
SU

In [ ]:
user_input =  decode(token_chunk)

message = get_message(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=True)

generated = assistant.generate(message=message)

corrected_generated_tokens, status = process_output(generated, token=token_chunk, allowed_labels=allowed_labels, assistant=assistant)


processed_chunks.append(corrected_generated_tokens)

#### Main processing function

In [ ]:
from src.models import get_message
from tqdm import tqdm

processed_chunks = []
for token_chunk in tqdm(token_chunks):

    user_input =  decode(token_chunk)

    message = get_message(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=True)

    generated = generate(messages=message, assistant=assistant)

    corrected_generated_tokens, status = process_output(generated, token=token_chunk, allowed_labels=allowed_labels, assistant=assistant)


    processed_chunks.append(corrected_generated_tokens)

100%|██████████| 18/18 [03:53<00:00, 12.99s/it]


#### Post Processing

In [149]:
from src.post_processing import chunks_to_html

output_html_content = chunks_to_html(processed_chunks, html_content)

   ✓ Flattened 18 chunks into 9689 tokens
   ✓ Merged to 12115 tokens
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Corrected 12425 tokens
   ✓ Brackets are coherent

✓ POST-PROCESSING COMPLETE
Final HTML length: 107451 characters



#### File saving

In [150]:
output_filename = Path("output") / f"{filename}_processed.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(output_html_content)

### For DEC1-3 

No chunking needed here, we just need the list of mention already labeled

#### Convert into tokens

In [23]:
from src import extract_body, tokenize, clean_tokens
tokens = tokenize(html_content)


#### Get already extracted mention

In [ ]:

from src.extractor import build_processing_segments
from src.extractor import get_list_of_mention
from src.models import get_messages
from tqdm import tqdm
from tqdm import tqdm

parent_mentions = get_list_of_mention(
        tokens=tokens,
        keep_labels=parents,
        label_type="auto_label"  # Process auto_labels from parent extraction
    )

print(f"Found {len(parent_mentions)} parent mentions to process")

segments = build_processing_segments(tokens, parent_mentions)

print(f"Built {len(segments)} token segments "
        f"({sum(s['process'] for s in segments)} to process)")

Found 23 parent mentions to process
Built 47 token segments (23 to process)


In [25]:
from src.prompts.prompt_utils import build_sublabel_definitions

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

from src.prompts.sublabel_definitions import SUBLABEL_DEFINITIONS_V2


if method in ["DEC1", "DEC2", "DEC3"]:
    sublabels_str = ", ".join(new_labels)
    sublabels_definition = build_sublabel_definitions(set(new_labels) - set(parents), sublabel_definitions=SUBLABEL_DEFINITIONS_V2)

    system_prompt = system_prompt.format(
            sublabels=sublabels_str,
            sublabels_definition=sublabels_definition,
        )

system_prompt used :  decomposed1-3.txt


#### Main processing function

In [ ]:
config = LabelTransformConfig(
    use_simplified=False,
    switch_type=False,
    keep_labels=already_labeled_labels,
    keep_attributes=["labelname"]
) # We only remove the attribute
 

failed_count = 0

for idx, segment in enumerate(tqdm(segments, desc="Processing mentions")):
        if not segment["process"]:
            continue


        mention = segment["tokens"]
        html_label = segment["meta"]["label"]

        
        prepared_tokens = prepare_label_tokens(mention, input_label_config)
        user_input = decode(prepared_tokens)

        filtered_fewshot = []
        for example in final_fewshot:
            if example[0].startswith(f"<{html_label.name}>"):
                filtered_fewshot.append(example)
        messages = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=filtered_fewshot, has_system_role=True)


        generated = assistant.generate(messages=messages)

        
        corrected_generated_tokens, status = process_output(generated=generated, token=prepare_label_tokens(mention, config), allowed_labels=allowed_labels, assistant=assistant, with_fallback=False)
        
        if not status.passed:
            failed_count += 1

        segment["tokens"] = corrected_generated_tokens

processed_tokens = [
        token
        for segment in segments
        for token in segment["tokens"]
    ]

Processing mentions: 100%|██████████| 47/47 [01:03<00:00,  1.36s/it]


#### Post Processing : tokens to HTML

In [28]:
from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
processed_html_content = tokens_to_html_after_decomposed1_3_prompting(processed_tokens, html_content)

   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)


#### Save File

In [29]:
output_filename = Path("output") / f"{filename}_processed2.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(processed_html_content)